<a href="https://colab.research.google.com/github/MarkChernov123/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-09-30%20%E2%80%94%20JSON%20to%20Tidy%20Tables%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## From one API response to a table you can join

Today's payload is a single response of the kind you will get on the midterm and capstone: metadata at the top, two levels of nesting, some records missing fields, and one list that somebody stored as a string.

You are producing two tables from it — one row per vendor, and one row per line item — and then joining them. Everything is guarded by row-count checks, because nested data hides lost rows better than anything else you will work with.

In [1]:
import json, pandas as pd

response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print(json.dumps(response, indent=2)[:400], '...')

{
  "game": "UVA vs Duke",
  "date": "2026-10-10",
  "attendance": 47200,
  "zones": [
    {
      "zone": "A",
      "capacity": 400,
      "vendors": [
        {
          "vendor_id": "V-01",
          "name": "Hoos Burgers",
          "sales": {
            "orders": 300,
            "revenue": 2250.0
          },
          "lines": [
            {
              "item": "Cheeseburger",
        ...


### Step 0 — map the shape before you touch it

Never normalize a payload you have not walked. These two cells tell you the nesting depth and how many rows to expect at each level, which is what your assertions will check against.

In [2]:
print('top-level keys:', list(response.keys()))
print('zones:', len(response['zones']))
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

top-level keys: ['game', 'date', 'attendance', 'zones']
zones: 3
vendors: 4
line items: 6


Four vendors, six line items. Write those numbers down — every step below gets checked against them.

**TODO:** what is one row of each table you are about to build?

- Table 1: one row per _..._
- Table 2: one row per _..._

### Build 1 — one row per vendor

**TODO:** flatten to one row per vendor. Carry down the game, the date, and the zone's own `zone` and `capacity`. Then rename the meta columns to something readable.

*Expected: 4 rows.*

In [3]:
# TODO: vendors = pd.json_normalize(response, record_path=..., meta=[...])
vendors = pd.json_normalize(
    response,
    record_path=['zones', 'vendors'],
    meta=[
        'game',
        'date',
        'attendance',
        ['zones', 'zone'],
        ['zones', 'capacity'],
    ],
)
# TODO: rename the zones.* columns
vendors = vendors.rename(
    columns={'zones.zone': 'zone', 'zones.capacity': 'capacity'}
)
# TODO: assert len(vendors) == 4

assert len(vendors) == 4
print('Vendor row count verified:', len(vendors))
print(vendors[['vendor_id', 'name', 'zone', 'capacity']])

Vendor row count verified: 4
  vendor_id             name zone capacity
0      V-01     Hoos Burgers    A      400
1      V-07     Stadium Dogs    A      400
2      V-12      Wahoo Wings    B      150
3      V-18  Rally Rain Gear    C      120


### Build 2 — find what the API did not tell you

One vendor is missing a field. **TODO:** print the null count per column, identify which vendor and which field, and add a boolean column that records whether revenue was reported.

Do **not** fill it with zero. Explain in a comment why that would be wrong here.

In [4]:
# TODO
print(vendors.isnull().sum())
vendors['revenue_reported'] = vendors['sales.revenue'].notna()

vendor_id        0
name             0
lines            0
sales.orders     0
sales.revenue    1
game             0
date             0
attendance       0
zone             0
capacity         0
dtype: int64


### Build 3 — one row per line item

**TODO:** flatten the same payload again, this time down to line items, keeping `vendor_id` and `zone` on every row.

*Expected: 6 rows, and the quantities should sum to 755.*

In [5]:
# TODO
lines = pd.json_normalize(
    response,
    record_path=['zones', 'vendors', 'lines'],
    meta=[
        'game',
        ['zones', 'zone'],
        ['zones', 'vendors', 'vendor_id'],
    ]
)
# TODO: assert len(lines) == 6
# TODO: assert lines['qty'].sum() == 755
assert len(lines) == 6
assert lines['qty'].sum() == 755

print("Line items row count verified:", len(lines))
print("Total quantity verified:", lines['qty'].sum())

Line items row count verified: 6
Total quantity verified: 755


### Build 4 — join the two tables

**TODO:** attach the vendor name and the zone capacity to each line item. Use `indicator=True` and confirm you still have 6 rows and nothing failed to match.

In [6]:
# TODO
print("Columns in lines:", lines.columns.tolist())
print("Columns in vendors:", vendors.columns.tolist())

# 2. Reset index just in case vendor_id got trapped in the index
lines_clean = lines.reset_index(drop=True)
vendors_clean = vendors.reset_index(drop=True)

# If lines still has the nested column name, rename it explicitly:
if 'zones.vendors.vendor_id' in lines_clean.columns:
    lines_clean = lines_clean.rename(columns={'zones.vendors.vendor_id': 'vendor_id'})

# 3. Perform the merge with indicator=True
merged_lines = lines_clean.merge(
    vendors_clean[['vendor_id', 'name', 'capacity']],
    on='vendor_id',
    how='inner',
    indicator=True
)

# 4. Confirm match and row counts
assert len(merged_lines) == 6
assert (merged_lines['_merge'] == 'both').all()

print("\nMerge successful! Breakdown:")
print(merged_lines['_merge'].value_counts())
print(merged_lines[['vendor_id', 'name', 'capacity', 'item', 'qty']])

Columns in lines: ['item', 'qty', 'game', 'zones.zone', 'zones.vendors.vendor_id']
Columns in vendors: ['vendor_id', 'name', 'lines', 'sales.orders', 'sales.revenue', 'game', 'date', 'attendance', 'zone', 'capacity', 'revenue_reported']

Merge successful! Breakdown:
_merge
both          6
left_only     0
right_only    0
Name: count, dtype: int64
  vendor_id             name capacity          item  qty
0      V-01     Hoos Burgers      400  Cheeseburger  180
1      V-01     Hoos Burgers      400       Hot Dog  120
2      V-07     Stadium Dogs      400       Hot Dog  210
3      V-12      Wahoo Wings      150     Wings 6pc   90
4      V-12      Wahoo Wings      150    Wings 12pc   60
5      V-18  Rally Rain Gear      120   Rain Poncho   95


### Build 5 — the list that came in as a string

A second source exported item lists as text, because someone wrote them to a CSV. **TODO:** turn this into one row per item — split the string, then `explode`.

In [7]:
csv_ish = pd.DataFrame({
    'order_id': [501, 502],
    'items': ['Cheeseburger|Soda|Fries', 'Rain Poncho'],
})
csv_ish

,order_id,items
0,501,Cheeseburger|Soda|Fries
1,502,Rain Poncho


In [8]:
# TODO: split on '|', explode, and clean up the index
csv_ish['item'] = csv_ish['items'].str.split('|')
exploded = csv_ish.explode('item').reset_index(drop=True)
# TODO: assert the result has 4 rows

exploded = exploded.drop(columns=['items'])

assert len(exploded) == 4

print("Row count verified:", len(exploded))
print(exploded)

Row count verified: 4
   order_id          item
0       501  Cheeseburger
1       501          Soda
2       501         Fries
3       502   Rain Poncho


### Build 6 — answer something

**TODO:** using your joined line-item table, print units sold per zone and the single best-selling item overall. Then say in one sentence what zone C's numbers suggest about the weather that day.

In [10]:
# TODO
units_per_zone = merged_lines.groupby('zones.zone')['qty'].sum()
print("Units sold per zone:")
print(units_per_zone)


best_item = merged_lines.groupby('item')['qty'].sum().idxmax()
best_item_qty = merged_lines.groupby('item')['qty'].sum().max()
print(f"\nBest-selling item overall: {best_item} ({best_item_qty} units)")

Units sold per zone:
zones.zone
A    510
B    150
C     95
Name: qty, dtype: int64

Best-selling item overall: Hot Dog (330 units)


---

## Checkpoint (participation)

Report your two row counts and what you did about the vendor with no revenue reported.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
vendor_rows = 4        # TODO: should be 4
line_item_rows = 6     # TODO: should be 6
units_total = 775        # TODO: should be 755
missing_revenue_call = 'Kept missing revenue as NaN and tracked with a boolean counter because filling with 0 would falsely imply zero sales'   # TODO: what you did and why

print('vendor rows:', vendor_rows)
print('line item rows:', line_item_rows)
print('total units:', units_total)
print('missing revenue:', missing_revenue_call)

vendor rows: 4
line item rows: 6
total units: 775
missing revenue: Kept missing revenue as NaN and tracked with a boolean counter because filling with 0 would falsely imply zero sales
